# 深度學習框架應用（大學部）｜深度學習基礎回顧（下）

這本筆記本配合投影片《深度學習基礎回顧（下）》。**練習一～五和投影片第 39～50 頁一題對一題**，每題 20%。

| 練習 | 題目 | 投影片 |
|---|---|---|
| 一 | 親手算損失 | 第 39 頁 |
| 二 | 梯度實算走三步 | 第 40 頁 |
| 三 | 用 autograd 驗證例題 | 第 41 頁 |
| 四 | 手刻線性迴歸訓練迴圈 | 第 44 頁 |
| 五 | 進階加分題 - 折幾次才對？ | 第 50 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」存一份自己的副本，之後都改自己那份。
2. 先跑「環境準備」，再由上往下做。標「參考程式」的格子跟投影片一字不差，先跑一次看結果再動手；標 ✏️ 的地方要你自己寫。
3. 這份都是小張量，CPU 就跑得動，不必切 GPU。改壞了就重新開原本的連結，再存一份複本。

**繳交**：檔名改成 `深度學習_W03下_學號_姓名.ipynb`，每題至少截 1 張圖，上傳到 Flipclass 課程平台。

> 程式碼的慣例：投影片沒有、但執行需要補上的那幾行，後面都標了 `# ←投影片未含，執行所需`。

## 環境準備

五題都會用到下面三個套件，先跑這一格。

In [ ]:
# ===== 環境準備 =====
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
print('PyTorch：', torch.__version__, '｜CUDA 可用：', torch.cuda.is_available())

## 練習一：親手算損失 (20%)

> 題目在投影片**第 39 頁**｜觀念在第 6～17 頁

**這題在練**：手算 MSE 與 CrossEntropy，並親眼看到「先做 softmax」為什麼會出事

**步驟**

1. pred = [2.5, 0.0, 2.0]、y = [3.0, −0.5, 2.0]，先手算平均平方誤差，再用 nn.MSELoss 驗證
2. 三類分類：logits = [2.0, 1.0, 0.1]、正確答案是第 0 類，用 nn.CrossEntropyLoss 算 loss
3. 故意先把 logits 做 softmax 再丟進 CrossEntropyLoss，比較兩個 loss 值
4. 讓 logits 可以求梯度，比較兩種寫法 backward 後梯度的大小，解釋為什麼 loss 會掉得很慢

**要繳交**（截圖上傳 Flipclass，每題至少 1 張）

- 四步的輸出截圖
- 第 4 步：一句話說明梯度為什麼變小

#### 投影片第 39 頁右邊的參考寫法

每行後面的註解，就是「為什麼這樣寫」。先跑一次看結果。

In [ ]:
import torch, torch.nn as nn
pred = torch.tensor([2.5, 0.0, 2.0])
y    = torch.tensor([3.0, -0.5, 2.0])
print(nn.MSELoss()(pred, y))           # 對手算答案
logits = torch.tensor([[2.0, 1.0, 0.1]]) # (1 筆, 3 類)
label  = torch.tensor([0])             # 整數類別
print(nn.CrossEntropyLoss()(logits, label))

#### ✏️ 動手做

步驟 1、2 就是上面的參考寫法。先手算 MSE，再看程式印出來的數字對不對。

✏️ **步驟 1：手算 MSE**

pred = [2.5, 0.0, 2.0]、y = [3.0, −0.5, 2.0]

- 三個誤差的平方：（　　）、（　　）、（　　）
- 平均 = （　　）

In [ ]:
# ✏️ 步驟 3：故意先做 softmax 再丟進 CrossEntropyLoss，和直接丟 logits 比一比
ce = nn.CrossEntropyLoss()
print('直接丟 logits  ：', ce(logits, label).item())
# print('先 softmax 再丟：', ce(torch.softmax(logits, dim=1), label).item())   # ✏️ 拿掉 # 跑跑看

In [ ]:
# ✏️ 步驟 4：兩種寫法 backward 之後，logits 的梯度有多大？
z = torch.tensor([[2.0, 1.0, 0.1]], requires_grad=True)
ce(z, label).backward()
print('直接丟 logits：梯度', z.grad, '，長度', round(z.grad.norm().item(), 4))

z.grad = None                            # 先把梯度清掉
# ✏️ 這次改成先 softmax 再丟，同樣 backward，印出梯度與長度

#### ✏️ 文字回答

**步驟 1 的手算過程**

（在這裡寫）

**步驟 4：先做 softmax 之後，梯度為什麼變小？這會讓訓練怎樣？**

（在這裡寫）

## 練習二：梯度實算走三步 (20%)

> 題目在投影片**第 40 頁**｜觀念在第 18～20 頁

**這題在練**：用 autograd 重現投影片上的三步下山，再親手試出學習率太大太小

**步驟**

1. 照右邊的寫法，讓 w 從 0 出發、lr = 0.1，用 autograd 走三步
2. 每步印出目前的 w 和梯度，和第 19 頁的表對答案（−6.00、−4.80、−3.84）
3. 把學習率換成 0.01 和 1.2，各走 10 步，記錄 w 怎麼變
4. 用一句話描述學習率太小、太大各會發生什麼事

**要繳交**（截圖上傳 Flipclass，每題至少 1 張）

- 三種學習率的輸出截圖
- 第 4 步的說明

#### 投影片第 40 頁右邊的參考寫法

每行後面的註解，就是「為什麼這樣寫」。先跑一次看結果。

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
for step in range(3):
    L = (w - 3) ** 2
    L.backward()                  # 算出 w.grad
    print(step + 1, w.item(), w.grad.item())
    with torch.no_grad():
        w -= 0.1 * w.grad         # 往下坡走一步
    w.grad.zero_()                # 擦掉，準備下一步

#### ✏️ 動手做

步驟 1、2 就是上面的參考寫法：三步印出來的數字，要和投影片的表一模一樣。

In [ ]:
# ✏️ 步驟 3：學習率換成 0.01 和 1.2，各走 10 步
def walk(lr, steps=10):
    w = torch.tensor(0.0, requires_grad=True)
    for step in range(steps):
        L = (w - 3) ** 2
        L.backward()
        with torch.no_grad():
            w -= lr * w.grad
        w.grad.zero_()
        print(f'lr = {lr}：第 {step + 1:2d} 步走完，w = {w.item():9.3f}')
    print()

walk(0.1, 3)                             # 先確認：0.600、1.080、1.464
# walk(0.01)                             # ✏️ 拿掉 # 跑跑看
# walk(1.2)

#### ✏️ 文字回答

**步驟 4：學習率太小會怎樣？太大會怎樣？（各一句話）**

（在這裡寫）

## 練習三：用 autograd 驗證例題 (20%)

> 題目在投影片**第 41 頁**｜參考程式第 42 頁｜逐行解說第 43 頁｜觀念在第 29～32 頁

**這題在練**：先手推例題的六個梯度，再用 autograd 對答案，順便看見 ReLU 傳 0

**步驟**

1. 先在紙上算第 29 頁的前向與六個梯度，不要先看答案
2. 用右邊的寫法把同樣的網路寫出來，z1、z2 要先 retain_grad() 才看得到梯度
3. backward 之後印出六個梯度，和手算對答案（−2、−1、−2、4、4、4）
4. 用 lr = 0.1 更新四個參數一步，確認得到 0.10、−0.40、−1.90、1.20
5. 把 w1 改成 −0.5、b2 改成 0 再跑一次，看 w1.grad、b1.grad 變成多少，用「ReLU 傳 0」解釋

**要繳交**（截圖上傳 Flipclass，每題至少 1 張）

- 手算過程的照片
- 程式輸出截圖
- 第 5 步的一句話說明

✏️ **步驟 1：先手算，不要先看答案**

x = 1、y = 1、w1 = 0.5、b1 = 0、w2 = −2、b2 = 1

| 前向 | 你算的 | 反向（梯度） | 你算的 |
|---|---|---|---|
| z1 = w1·x + b1 |  | ∂L/∂z2 |  |
| a1 = ReLU(z1) |  | ∂L/∂w2 |  |
| z2 = w2·a1 + b2 |  | ∂L/∂b2 |  |
| L = (z2 − y)² |  | ∂L/∂z1 |  |
|  |  | ∂L/∂w1 |  |
|  |  | ∂L/∂b1 |  |

#### 參考程式（投影片第 42 頁，`autograd_basics.py`）

> 投影片說明：y.item() 會印出 13。若把 requires_grad 拿掉再呼叫 backward()，你會看到 RuntimeError —— 這個訊息之後很常見。

In [ ]:
import torch
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)

y = w * x ** 2 + 1            # 前向：y = w·x² + 1
print(y.item(), y.grad_fn)    # grad_fn 記錄產生 y 的運算

y.backward()                  # 從純量出發，自動套鏈鎖律
print(x.grad)                 # dy/dx = 2wx = 12
print(w.grad)                 # dy/dw = x²  = 4

x.grad.zero_(); w.grad.zero_()   # 梯度預設累加，記得清空
with torch.no_grad():            # 推論時不建圖，省記憶體
    z = w * x ** 2

#### 投影片第 41 頁右邊的參考寫法

每行後面的註解，就是「為什麼這樣寫」。先跑一次看結果。

In [ ]:
x, y = torch.tensor(1.0), torch.tensor(1.0)
w1, b1, w2, b2 = [torch.tensor(v, requires_grad=True)
                  for v in (0.5, 0.0, -2.0, 1.0)]
z1 = w1 * x + b1;  z1.retain_grad()  # 中間值要留梯度
z2 = w2 * torch.relu(z1) + b2;  z2.retain_grad()
L = (z2 - y) ** 2
L.backward()
print(z2.grad, w2.grad, b2.grad, z1.grad, w1.grad, b1.grad)

#### ✏️ 動手做

步驟 2、3 就是上面的參考寫法：印出來的六個數字，和你步驟 1 的表對答案。

In [ ]:
# ✏️ 步驟 4：用 lr = 0.1 更新四個參數一步（要接在上面的參考寫法之後跑）
with torch.no_grad():
    for p in (w1, b1, w2, b2):
        pass                             # ✏️ 往下坡走一步，取代 pass（提示：p -= ...）
print(w1.item(), b1.item(), w2.item(), b2.item())   # 要是 0.10、-0.40、-1.90、1.20

In [ ]:
# ✏️ 步驟 5：w1 改成 -0.5、b2 改成 0，重跑一次前向與反向
x, y = torch.tensor(1.0), torch.tensor(1.0)
w1, b1, w2, b2 = [torch.tensor(v, requires_grad=True)
                  for v in (0.5, 0.0, -2.0, 1.0)]    # ✏️ 改這一行的數字
z1 = w1 * x + b1
z2 = w2 * torch.relu(z1) + b2
L = (z2 - y) ** 2
L.backward()
print('z1 =', z1.item(), '｜w1.grad =', w1.grad.item(),
      '｜b1.grad =', b1.grad.item(), '｜b2.grad =', b2.grad.item())

#### ✏️ 文字回答

**步驟 3：六個梯度和手算一樣嗎？不一樣的話，是哪一步算錯？**

（在這裡寫）

**步驟 5：w1.grad、b1.grad 變成多少？用「ReLU 傳 0」解釋。**

（在這裡寫）

## 練習四：手刻線性迴歸訓練迴圈 (20%)

> 題目在投影片**第 44 頁**｜參考程式第 46 頁｜逐行解說第 47、48 頁｜觀念在第 33～34 頁

**這題在練**：自己寫出五步驟迴圈，並用 loss 曲線看懂學習率與清梯度

**步驟**

1. 照參考程式造資料（真值 w = 3、b = 2），寫出五步驟迴圈跑 200 輪
2. 印出最後的 w、b，確認接近 3 和 2
3. 每輪把 loss 存進 list，用 matplotlib 畫出 loss 曲線
4. 學習率改成 1.0 和 0.001 各跑一次，三條 loss 曲線畫在同一張圖
5. 把清梯度那行註解掉，印出前 5 輪的 w.grad，說明為什麼會越來越大

**要繳交**（截圖上傳 Flipclass，每題至少 1 張）

- 三條 loss 曲線的圖
- 第 5 步的輸出與說明

#### 參考程式（投影片第 46 頁，`linreg_autograd.py`）

> 投影片說明：訓練後 w 應該非常接近 3、b 接近 2。這裡刻意不用 nn.Linear 與 optimizer，是為了讓你看清楚框架幫你包掉了哪幾行 —— 之後的主題就會換成標準寫法。

In [ ]:
import torch
torch.manual_seed(0)
x = torch.linspace(-1, 1, 100).unsqueeze(1)   # (100,1)
y = 3 * x + 2 + 0.1 * torch.randn_like(x)     # 真值 w=3, b=2
w = torch.zeros(1, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1

for epoch in range(200):
    pred = x @ w + b                  # 1) 前向傳播
    loss = ((pred - y) ** 2).mean()   # 2) 計算 MSE 損失
    loss.backward()                   # 3) 反向求梯度
    with torch.no_grad():             # 4) 更新時不要建圖
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()    # 5) 清空梯度

print(w.item(), b.item(), loss.item())

#### 投影片第 44 頁右邊的參考寫法

每行後面的註解，就是「為什麼這樣寫」。先跑一次看結果。

In [ ]:
w = torch.zeros(1, 1, requires_grad=True)  # ←投影片未含，執行所需：從 0 重新開始
b = torch.zeros(1, requires_grad=True)  # ←投影片未含，執行所需
losses = []
for epoch in range(200):
    loss = ((x @ w + b - y) ** 2).mean()
    losses.append(loss.item())      # 存下來畫圖
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad; b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()  # 第 5 步拿掉這行

#### ✏️ 動手做

步驟 1、2 就是參考程式：最後印出的 w、b 要接近 3 和 2。上面的參考寫法把每輪的 loss 存進 `losses`，下面接著畫圖。

In [ ]:
# 步驟 3：把 losses 畫成曲線
plt.plot(losses)
plt.xlabel('epoch'); plt.ylabel('MSE loss'); plt.yscale('log')
plt.grid(alpha=0.3); plt.show()

In [ ]:
# ✏️ 步驟 4：學習率 0.1、1.0、0.001 各跑 200 輪，三條 loss 畫在同一張圖
def train(lr, epochs=200):
    w = torch.zeros(1, 1, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    hist = []
    for epoch in range(epochs):
        loss = ((x @ w + b - y) ** 2).mean()
        hist.append(loss.item())
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad; b -= lr * b.grad
        w.grad.zero_(); b.grad.zero_()
    return hist

plt.plot(train(0.1), label='lr = 0.1')
# ✏️ 再加 lr = 1.0 和 lr = 0.001 兩條
plt.xlabel('epoch'); plt.ylabel('MSE loss'); plt.yscale('log')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# ✏️ 步驟 5：把清梯度那行拿掉，印出前 5 輪的 w.grad
w = torch.zeros(1, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
for epoch in range(5):
    loss = ((x @ w + b - y) ** 2).mean()
    loss.backward()
    print(f'第 {epoch + 1} 輪  w.grad = {w.grad.item():8.4f}')
    with torch.no_grad():
        w -= lr * w.grad; b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()       # ✏️ 在這行最前面加上 # 再跑一次，比比看

#### ✏️ 文字回答

**步驟 4：lr = 1.0 和 lr = 0.001 的曲線各長什麼樣？**

（在這裡寫）

**步驟 5：拿掉清梯度之後，w.grad 為什麼越來越大？**

（在這裡寫）

## 練習五：進階加分題 - 折幾次才對？ (20%)

> 題目在投影片**第 50 頁**｜觀念在（上）第 26～27 頁

**這題在練**：用同一份資料訓練大小不同的網路，自己判斷欠擬合、剛好、過擬合

**步驟**

1. 造資料：x 取 −3～3 之間 200 個點，y = sin(x) 加一點雜訊，隨機切成訓練 150、驗證 50
2. 用 nn.Sequential 建 1 → n → 1 的網路，optimizer 先照抄 Adam（之後的主題會正式講）
3. n = 2、16、256 各訓練 2000 輪，每 100 輪記錄訓練與驗證 loss
4. 畫出三組 loss 曲線和擬合結果，照（上）第 27 頁的診斷表判斷各屬於哪一種
5. 看不到過擬合的話，把訓練資料減到 20 筆、雜訊加到 0.3 再試，說明為什麼資料少更容易死背

**要繳交**（截圖上傳 Flipclass，每題至少 1 張）

- 三組 loss 曲線與擬合結果的圖
- 判斷結果與理由

#### 投影片第 50 頁右邊的參考寫法

每行後面的註解，就是「為什麼這樣寫」。先跑一次看結果。

In [ ]:
n = 16  # ←投影片未含，執行所需：先試 16 個神經元
x = torch.linspace(-3, 3, 200).unsqueeze(1)
y = torch.sin(x) + 0.1 * torch.randn_like(x)
idx = torch.randperm(200)          # 打亂再切
tr, va = idx[:150], idx[150:]
net = nn.Sequential(nn.Linear(1, n), nn.ReLU(),
                    nn.Linear(n, 1))
opt = torch.optim.Adam(net.parameters(), lr=0.01)
for epoch in range(2000):
    loss = ((net(x[tr]) - y[tr]) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

#### ✏️ 動手做

上面的參考寫法只訓練一個網路、也沒記錄驗證 loss。下面三格把它包成函式，一次比三種大小。

In [ ]:
# ✏️ 步驟 1～3：造資料、切訓練／驗證，n = 2、16、256 各訓練 2000 輪
def fit(n, x_tr, y_tr, x_va, y_va, epochs=2000, seed=0):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(1, n), nn.ReLU(), nn.Linear(n, 1))
    opt = torch.optim.Adam(net.parameters(), lr=0.01)   # 最佳化器之後的主題會正式講，先照抄
    hist = []
    for epoch in range(epochs):
        loss = ((net(x_tr) - y_tr) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if (epoch + 1) % 100 == 0:       # 每 100 輪記一次訓練與驗證 loss
            with torch.no_grad():
                val = ((net(x_va) - y_va) ** 2).mean()
            hist.append((epoch + 1, loss.item(), val.item()))
    return net, hist

def run(sizes, n_train=150, noise=0.1, seed=0):
    torch.manual_seed(seed)
    x = torch.linspace(-3, 3, 200).unsqueeze(1)
    y = torch.sin(x) + noise * torch.randn_like(x)
    idx = torch.randperm(200)
    tr, va = idx[:n_train], idx[150:]    # 驗證固定是最後 50 筆
    out = {}
    for n in sizes:
        out[n] = fit(n, x[tr], y[tr], x[va], y[va])
        _, h = out[n]
        print(f'n = {n:3d}   訓練 loss {h[-1][1]:.4f}   驗證 loss {h[-1][2]:.4f}')
    return out, (x, y, tr, va)

results, data = run((2,))                # ✏️ 改成 (2, 16, 256)

In [ ]:
# 步驟 4：畫出每個網路的 loss 曲線（左）與擬合結果（右）
def show(results, data):
    x, y, tr, va = data
    xs = torch.linspace(-3, 3, 400).unsqueeze(1)
    fig, axes = plt.subplots(len(results), 2, figsize=(12, 3.2 * len(results)), squeeze=False)
    for row, (n, (net, hist)) in enumerate(results.items()):
        ep, trl, val = zip(*hist)
        ax = axes[row, 0]
        ax.plot(ep, trl, label='train'); ax.plot(ep, val, label='val')
        ax.set_yscale('log'); ax.set_title(f'n = {n}: loss'); ax.legend()
        ax = axes[row, 1]
        ax.scatter(x[tr], y[tr], s=10, label='train'); ax.scatter(x[va], y[va], s=10, label='val')
        with torch.no_grad():
            ax.plot(xs, net(xs), 'k', lw=2, label='model')
        ax.set_title(f'n = {n}: fit'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

show(results, data)

In [ ]:
# ✏️ 步驟 5：看不出過擬合的話，訓練資料減到 20 筆、雜訊加到 0.3 再試
results2, data2 = run((2, 16, 256), n_train=150, noise=0.1)   # ✏️ 改成 n_train=20, noise=0.3
show(results2, data2)

#### ✏️ 文字回答

**步驟 4：n = 2、16、256 各屬於欠擬合、剛好、過擬合的哪一種？理由是什麼？（對照投影片的診斷表）**

（在這裡寫）

**步驟 5：為什麼資料少、雜訊多，大網路更容易「死背」？**

（在這裡寫）